In [0]:
from pyspark.sql import functions as F
from datetime import datetime as dt
import uuid

SILVER = "/Volumes/workspace/olist/silver"
GOLD   = "/Volumes/workspace/olist/gold"

EXECUCAO_ID = str(uuid.uuid4())

# Lê as 4 tabelas Gold
df_faturamento = spark.read.format("delta").load(f"{GOLD}/faturamento_por_estado")
df_categoria   = spark.read.format("delta").load(f"{GOLD}/performance_por_categoria")
df_entregas    = spark.read.format("delta").load(f"{GOLD}/analise_entregas")
df_clientes = spark.read.format("delta").load(f"{GOLD}/top_clientes_ltv")

# Lê o silver como fonte da verdade
df_silver = spark.read.format("delta").load(f"{SILVER}/orders_enriched")

print(f"Execução ID: {EXECUCAO_ID}")
print(f"\nTabelas Gold carregadas:")
print(f"  faturamento_por_estado:    {df_faturamento.count()} registros")
print(f"  performance_por_categoria: {df_categoria.count()} registros")
print(f"  analise_entregas:          {df_entregas.count()} registros")
print(f"  top_clientes_ltv:          {df_clientes.count()} registros")
print(f"\nSilver (fonte da verdade):   {df_silver.count()} registros")

In [0]:
problemas = []

# Silver filtrando só com delivered - mesma base do Gold
df_silver_delivered = df_silver.filter(F.col("order_status") == "delivered")
total_silver_delivered = df_silver_delivered.count()

# CHECK 1 - Receita total do Gold deve bater com o Silver
receita_silver = df_silver_delivered \
    .agg(F.round(F.sum("price"), 2).alias("receita")) \
    .collect()[0]["receita"]

receita_gold = df_faturamento \
    .agg(F.round(F.sum("receita_produtos"), 2).alias("receita")) \
    .collect()[0]["receita"]

divergencia_receita = abs(receita_silver - receita_gold)

print(f"Receita Silver (delivered): R$ {receita_silver:,.2f}")
print(f"Receita Gold:               R$ {receita_gold:,.2f}")
print(f"Divergência:                R$ {divergencia_receita:,.2f}")

if divergencia_receita > 0.01:
    problemas.append(f"Receita diverge em R$ {divergencia_receita:,.2f}")

# CHECK 2 - Total de estados no Gold deve cobrir todos os estados do Silver

estados_silver = df_silver_delivered.select("customer_state").distinct().count()
estados_gold   = df_faturamento.count()

if estados_silver != estados_gold:
    problemas.append(f"Estados no Gold ({estados_gold}) diferente do Silver ({estados_silver})")

# CHECK 3 - Total de clientes únicos no Gold deve bater com o silver

clientes_silver = df_silver_delivered.select("customer_unique_id").distinct().count()
clientes_gold = df_clientes.count()

print(f"\nClientes únicos Silver: {clientes_silver}")
print(f"Clientes únicos Gold:   {clientes_gold}")

if clientes_silver != clientes_gold:
    problemas.append(f"Clientes no Gold ({clientes_gold}) diferente do Silver ({clientes_silver})")


print(f"\nProblemas encontrados: {len(problemas)}")
for p in problemas:
    print(f"  ❌ {p}")

if not problemas:
    print("  ✅ Todas as consistências validadas")

In [0]:
problemas = []

# CHECK 1 - faturamento_por_estado
# Nenhum estado pode ter receita zero ou negativa
receita_zero = df_faturamento.filter(F.col("receita_total") <= 0).count()
ticket_suspeito = df_faturamento.filter(F.col("ticket_medio") < 1).count()

print(f"faturamento_por_estado:")
print(f" Receita zero ou negativa: {receita_zero}")
print(f" Ticket médio abaixo R$1:   {ticket_suspeito}")

if receita_zero > 0:
    problemas.append(f"faturamento_por_estado: {receita_zero} estados com receita zero ou negativa")
if ticket_suspeito > 0:
    problemas.append(f"faturamento_por_estado: {ticket_suspeito} estados com ticket médio abaixo de R$1")

# CHECK 2 - performance_por_categoria
# Nenhuma categoria pode ter receita zero ou preço médio negativa
cat_receita_zero = df_categoria.filter(F.col("receita_total") <= 0).count()
cat_preco_suspeito = df_categoria.filter(F.col("preco_medio") < 1).count()

print(f"\nperformance_por_categoria:")
print(f" Receita zero ou negativa: {cat_receita_zero}")
print(f" Preço médio abaixo R$1:   {cat_preco_suspeito}")

if cat_receita_zero > 0:
    problemas.append(f"performance_por_categoria: {cat_receita_zero} categorias com receita zero ou negativa")
if cat_preco_suspeito > 0:
    problemas.append(f"performance_por_categoria: {cat_preco_suspeito} categorias com preço médio abaixo de R$1")

# CHECK 3 - analise_entregas
# Prazo médio negativo é impossível
# Prazo médio acima de 60 dias é suspeito
prazo_negativo = df_entregas.filter(F.col("prazo_medio_dias") < 0).count()
prazo_suspeito = df_entregas.filter(F.col("prazo_medio_dias") > 60).count()
prazo_min_negativo = df_entregas.filter(F.col("prazo_minimo_dias") < 0).count()

print(f"\nanalise_entregas:")
print(f" Prazo médio negativo:   {prazo_negativo}")
print(f" Prazo médio acima de 60 dias: {prazo_suspeito}")
print(f" Prazo mínimo negativo:  {prazo_min_negativo}")

if prazo_negativo > 0:
    problemas.append(f"analise_entregas: {prazo_negativo} prazos médios negativos")
if prazo_suspeito > 0:
    problemas.append(f"analise_entregas: {prazo_suspeito} prazos médios acima de 60 dias")
if prazo_min_negativo > 0:
    problemas.append(f"analise_entregas: {prazo_min_negativo} prazos mínimos negativos")

# CHECK 4 - top_clientes_ltv
# LTV total nunca pode ser menor que receita total
# Total cliente deve ter pelo menos 1 pedido
ltv_menor_receita = df_clientes.filter(F.col("ltv_total") < F.col("receita_total")).count()
sem_pedidos = df_clientes.filter(F.col("total_pedidos") < 1).count()

print(f"\ntop_clientes_ltv:")
print(f" LTV total menor que receita total: {ltv_menor_receita}")
print(f" Clientes sem pedidos: {sem_pedidos}")

if ltv_menor_receita > 0:
    problemas.append(f"top_clientes_ltv: {ltv_menor_receita} clientes com LTV total menor que receita total")
if sem_pedidos > 0:
    problemas.append(f"top_clientes_ltv: {sem_pedidos} clientes sem pedidos")
print(f"\nProblemas encontrados: {len(problemas)}")
for p in problemas:
    print(f"  ❌ {p}")
if not problemas:
    print("  ✅ Todas as consistências validadas")


In [0]:
import time
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, TimestampType)
from datetime import datetime as dt

LOG_PATH   = "/Volumes/workspace/olist/bronze/_log"
LOG_SCHEMA = StructType([
    StructField("execucao_id",        StringType(),    nullable=False),
    StructField("tabela",             StringType(),    nullable=False),
    StructField("status",             StringType(),    nullable=False),
    StructField("registros_lidos",    LongType(),      nullable=True),
    StructField("registros_gravados", LongType(),      nullable=True),
    StructField("divergencia_pct",    DoubleType(),    nullable=True),
    StructField("tempo_segundos",     DoubleType(),    nullable=True),
    StructField("mensagem",           StringType(),    nullable=True),
    StructField("timestamp",          TimestampType(), nullable=False)
])

def gravar_log(execucao_id, tabela, status, registros_lidos=None,
               registros_gravados=None, divergencia_pct=None,
               tempo_segundos=None, mensagem=None):
    linha = [{
        "execucao_id":        execucao_id,
        "tabela":             tabela,
        "status":             status,
        "registros_lidos":    registros_lidos,
        "registros_gravados": registros_gravados,
        "divergencia_pct":    divergencia_pct,
        "tempo_segundos":     tempo_segundos,
        "mensagem":           mensagem,
        "timestamp":          dt.now()
    }]
    spark.createDataFrame(linha, schema=LOG_SCHEMA) \
        .write.format("delta").mode("append").save(LOG_PATH)

def processar_gold(execucao_id: str):
    inicio = time.time()
    problemas = []

    print(f"\n{'='*50}")
    print(f"Validando camada Gold")
    print(f"{'='*50}")

    try:
        # Lê todas as tabelas
        df_fat      = spark.read.format("delta").load(f"{GOLD}/faturamento_por_estado")
        df_cat      = spark.read.format("delta").load(f"{GOLD}/performance_por_categoria")
        df_ent      = spark.read.format("delta").load(f"{GOLD}/analise_entregas")
        df_cli      = spark.read.format("delta").load(f"{GOLD}/top_clientes_ltv")
        df_silver   = spark.read.format("delta").load(f"{SILVER}/orders_enriched")
        df_delivered = df_silver.filter(F.col("order_status") == "delivered")

        total_registros = df_fat.count() + df_cat.count() + df_ent.count() + df_cli.count()

        # Consistência Silver x Gold
        receita_silver = df_delivered.agg(F.round(F.sum("price"), 2)).collect()[0][0]
        receita_gold   = df_fat.agg(F.round(F.sum("receita_produtos"), 2)).collect()[0][0]
        if abs(receita_silver - receita_gold) > 0.01:
            problemas.append(f"Divergência de receita: R$ {abs(receita_silver - receita_gold):.2f}")

        estados_silver = df_delivered.select("customer_state").distinct().count()
        if estados_silver != df_fat.count():
            problemas.append(f"Estados divergentes: Silver {estados_silver} vs Gold {df_fat.count()}")

        clientes_silver = df_delivered.select("customer_unique_id").distinct().count()
        clientes_gold   = df_cli.select("customer_unique_id").distinct().count()
        if clientes_silver != clientes_gold:
            problemas.append(f"Clientes divergentes: Silver {clientes_silver} vs Gold {clientes_gold}")

        # Validações de negócio
        if df_fat.filter(F.col("receita_total") <= 0).count() > 0:
            problemas.append("faturamento_por_estado: receita zero ou negativa")
        if df_cat.filter(F.col("receita_total") <= 0).count() > 0:
            problemas.append("performance_por_categoria: receita zero ou negativa")
        if df_ent.filter(F.col("prazo_medio_dias") < 0).count() > 0:
            problemas.append("analise_entregas: prazo médio negativo")
        if df_ent.filter(F.col("prazo_medio_dias") > 60).count() > 0:
            problemas.append("analise_entregas: prazo médio acima de 60 dias")
        if df_cli.filter(F.col("ltv_total") < F.col("receita_total")).count() > 0:
            problemas.append("top_clientes_ltv: LTV menor que receita")

        tempo  = round(time.time() - inicio, 2)
        status = "SUCCESS" if not problemas else "FAILED"
        mensagem = f"Processado com sucesso em {tempo}s" if not problemas else " | ".join(problemas)

        gravar_log(
            execucao_id        = execucao_id,
            tabela             = "gold",
            status             = status,
            registros_lidos    = total_registros,
            registros_gravados = total_registros if status == "SUCCESS" else None,
            tempo_segundos     = tempo,
            mensagem           = mensagem
        )

        print(f"  Status: {status} | Tempo: {tempo}s")
        if problemas:
            for p in problemas:
                print(f"  ❌ {p}")
        else:
            print(f"  ✅ {total_registros} registros validados")

    except Exception as e:
        tempo = round(time.time() - inicio, 2)
        gravar_log(
            execucao_id    = execucao_id,
            tabela         = "gold",
            status         = "FAILED",
            tempo_segundos = tempo,
            mensagem       = f"Erro inesperado: {str(e)}"
        )
        print(f"  ERRO INESPERADO: {e}")

print("Orquestrador Gold carregado!")

In [0]:
gravar_log(
    execucao_id = EXECUCAO_ID,
    tabela      = "PIPELINE_GOLD",
    status      = "STARTED",
    mensagem    = "Início da blindagem do Gold"
)

processar_gold(EXECUCAO_ID)

# Resumo da execução
df_resumo = spark.read \
    .format("delta") \
    .load(LOG_PATH) \
    .filter(F.col("execucao_id") == EXECUCAO_ID) \
    .filter(F.col("tabela") != "PIPELINE_GOLD")

print(f"\n{'='*50}")
print(f"RESUMO DA EXECUÇÃO — {EXECUCAO_ID}")
print(f"{'='*50}")

df_resumo.select(
    "tabela",
    "status",
    "registros_lidos",
    "registros_gravados",
    "tempo_segundos",
    "mensagem"
).show(truncate=False)

falhas   = df_resumo.filter(F.col("status") == "FAILED").count()
sucessos = df_resumo.filter(F.col("status") == "SUCCESS").count()

gravar_log(
    execucao_id = EXECUCAO_ID,
    tabela      = "PIPELINE_GOLD",
    status      = "FINISHED" if falhas == 0 else "FINISHED_WITH_ERRORS",
    mensagem    = f"Concluído — SUCCESS: {sucessos} | FAILED: {falhas}"
)

print(f"\nSUCCESS: {sucessos} | FAILED: {falhas}")
print(f"Pipeline Gold finalizado!")